# Tutorial: Weights & Biases (W&B)

Do zero ao primeiro experimento rastreado.

**Antes de começar:**

1. Crie uma conta gratuita em [wandb.ai](https://wandb.ai)
2. Rode no terminal:
```
pip install wandb scikit-learn numpy
wandb login
```
O `wandb login` pede sua API key — copie de [wandb.ai/authorize](https://wandb.ai/authorize).

## O que é o W&B?

Uma plataforma de rastreamento de experimentos de machine learning.
Resolve o problema clássico: *"qual combinação de hiperparâmetros gerou aquele resultado bom da semana passada?"*

Os 3 conceitos centrais:

| Função | O que faz |
|---|---|
| `wandb.init()` | Abre uma **run** (uma execução rastreada) |
| `config` | Registra os hiperparâmetros usados |
| `wandb.log()` | Envia métricas ao longo do treino |

Tudo aparece automaticamente em dashboards no site.

## Importações

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
import time

# a única dependência nova
import wandb

## 1. `wandb.init()` — abrir uma run

Uma **run** é uma execução única do seu experimento. Cada vez que você roda esta célula, uma nova run aparece no painel do W&B.

Parâmetros que vale conhecer:

| Parâmetro | Função |
|---|---|
| `project` | Nome do projeto (agrupa suas runs) |
| `config` | Dicionário com hiperparâmetros |
| `notes` | Anotação livre para você lembrar o que testou |
| `tags` | Etiquetas para filtrar runs depois |

In [ ]:
config = {
    "learning_rate": 0.01,       # taxa de aprendizado do otimizador
    "epochs": 10,                # passadas completas pelos dados
    "batch_size": 32,            # amostras por passo de atualização
    "hidden_layer_size": 64,     # neurônios na camada oculta
    "test_size": 0.2,            # fração dos dados para teste
    "random_state": 42,          # semente para reprodutibilidade
}

run = wandb.init(
    project="tutorial-wandb",
    config=config,
    notes="Primeira run — baseline com MLP no digits",
    tags=["tutorial", "baseline"],
)

# o W&B imprime um link aqui — clique para ver a run no navegador
print(f"Run aberta: {run.url}")

## 2. Preparar os dados

Usamos o dataset `digits` do sklearn: 1797 imagens 8×8 de dígitos (0–9). Não precisa baixar nada.

In [ ]:
digits = load_digits()
X, y = digits.data, digits.target

print(f"Amostras: {X.shape[0]}")
print(f"Dimensões por amostra: {X.shape[1]} (imagem 8x8 achatada)")
print(f"Classes: {np.unique(y)}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=config["test_size"],
    random_state=config["random_state"],
)

# pixels vão de 0 a 16 — dividimos por 16 para normalizar entre 0 e 1
# isso estabiliza o gradiente durante o treino
X_train = X_train / 16.0
X_test = X_test / 16.0

print(f"Treino: {X_train.shape[0]} amostras | Teste: {X_test.shape[0]} amostras")

## 3. Criar o modelo

`MLPClassifier` é um Perceptron Multicamada — rede neural densa simples.

Detalhe importante: usamos `warm_start=True` para poder chamar `.fit()` várias vezes sem resetar os pesos. Isso simula um loop de épocas manual, que é o que precisamos para logar métricas época a época.

In [ ]:
model = MLPClassifier(
    hidden_layer_sizes=(config["hidden_layer_size"],),
    learning_rate_init=config["learning_rate"],
    batch_size=config["batch_size"],
    max_iter=1,           # 1 iteração por chamada de .fit()
    warm_start=True,      # mantém pesos entre chamadas
    verbose=False,
    random_state=config["random_state"],
)

## 4. `wandb.log()` — treinar e registrar métricas

`wandb.log()` aceita qualquer dicionário `{nome_da_métrica: valor}`. Cada chamada vira um ponto nos gráficos do painel.

Dica: prefixos como `train/` e `test/` agrupam as métricas automaticamente no painel.

In [ ]:
start = time.time()

for epoch in range(config["epochs"]):
    model.fit(X_train, y_train)

    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc = accuracy_score(y_test, model.predict(X_test))
    loss = model.loss_  # cross-entropy interna do sklearn

    # cada chamada de log() = um ponto no gráfico
    wandb.log({
        "epoch": epoch + 1,
        "train/accuracy": train_acc,
        "test/accuracy": test_acc,
        "train/loss": loss,
    })

    print(
        f"Época {epoch + 1:>2}/{config['epochs']} | "
        f"loss: {loss:.4f} | "
        f"acc treino: {train_acc:.4f} | "
        f"acc teste: {test_acc:.4f}"
    )

train_time = time.time() - start
print(f"\nTempo de treino: {train_time:.2f}s")

## 5. `wandb.summary` — métricas finais

`wandb.summary` é um dicionário especial que aparece na tabela de comparação entre runs. Diferente do `wandb.log()` que registra séries temporais, o `summary` guarda um valor único por métrica — o resultado final do experimento.

In [ ]:
wandb.summary["final_train_accuracy"] = train_acc
wandb.summary["final_test_accuracy"] = test_acc
wandb.summary["training_time_seconds"] = train_time

## 6. `wandb.Table` — dados tabulares (bônus)

Tabelas permitem inspecionar predições individuais direto no painel do W&B. Útil para análise de erros.

In [ ]:
y_pred = model.predict(X_test)

tabela = wandb.Table(
    columns=["indice", "valor_real", "valor_predito", "acertou"]
)

for i in range(min(100, len(y_test))):
    tabela.add_data(i, int(y_test[i]), int(y_pred[i]), y_test[i] == y_pred[i])

# aparece como aba interativa no painel da run
wandb.log({"predicoes_teste": tabela})

## 7. `wandb.finish()` — encerrar a run

Sempre chame `finish()` no final. Isso envia dados pendentes no buffer, marca a run como finalizada no painel e libera recursos.

In [ ]:
wandb.finish()

print("Pronto! Acesse o link do início para ver os gráficos.")

## Exercícios

Agora que você tem uma run baseline, experimente:

**Mude hiperparâmetros** no dicionário `config` e rode tudo de novo. Cada execução cria uma nova run — compare no painel.

**Adicione uma camada oculta:** troque `hidden_layer_sizes=(64,)` por `(64, 32)` e veja se melhora.

**Teste outro learning_rate:** 0.001, 0.01, 0.1. No painel, agrupe as runs por `config.learning_rate` para comparar.

---

**Referências:**
- [Documentação W&B](https://docs.wandb.ai)
- [wandb.init()](https://docs.wandb.ai/ref/python/init)
- [wandb.log()](https://docs.wandb.ai/ref/python/log)
- [wandb.Table](https://docs.wandb.ai/guides/tables)